# 音のプログラミング 第7回: 最終プロジェクト — 既存曲の再現

## このレッスンの目標

好きな曲を1曲選び、レッスン1〜6で学んだ技術を総動員してプログラムで再現します。

### 評価のポイント
- **技術要素の活用度**: 波形生成、エンベロープ、エフェクト、シーケンサーなどをどれだけ活用できたか
- **原曲の再現度**: メロディ、リズム、コード進行がどれだけ正確か
- **コードの品質**: 読みやすく整理されたコードになっているか

### レッスンの流れ
1. まず2つの例題で「既存曲の再現」の手順を学ぶ
2. 楽譜からコードへの変換方法を確認する
3. 音色設計のリファレンスを参照する
4. 自分で選んだ曲を実装する

**所要時間:** 90分

## 🛠️ 環境セットアップ

In [ ]:
# 🛠️ 環境セットアップ
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    !pip install japanize-matplotlib
    !git clone https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
    import japanize_matplotlib
else:
    print("🔧 ローカル環境を設定中...")
    sys.path.append('..')
    import platform
    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'Hiragino Sans'
    else:
        plt.rcParams['font.family'] = 'Meiryo'

print("✅ セットアップ完了")

## 🛠️ 今回使用するライブラリ

In [ ]:
from audio_lib import (
    sine_wave, square_wave, sawtooth_wave, triangle_wave,
    adsr, save_audio, AudioSignal,
    note_to_frequency, note_name_to_number, number_to_note_name,
    Sequencer, Note, Track, create_simple_melody, create_chord,
    BasicPiano, BasicOrgan, BasicGuitar, BasicDrum,
    LowPassFilter, HighPassFilter,
    Reverb, Delay, Chorus,
)
from audio_lib.notebook import play_sound

sample_rate = 44100

## 🎯 例題1：「きらきら星」をリッチに仕上げる

段階的に音を豊かにしていく手順を学びます。

### ステップ1：メロディだけを打ち込む

In [ ]:
seq = Sequencer()
melody_track = Track(name="melody")

melody_notes = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody_track, melody_notes, note_duration=0.5)
seq.add_track(melody_track)
seq.set_instrument("melody", BasicPiano())

audio = seq.render()
display(play_sound(audio, "ステップ1：メロディだけ（ピアノ）"))

### ステップ2：コード伴奏を追加する

In [ ]:
seq = Sequencer()

melody_track = Track(name="melody")
melody_notes = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody_track, melody_notes, note_duration=0.5)

chord_track = Track(name="chords")
create_chord(chord_track, ["C3", "E3", "G3"], start_time=0.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=1.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=2.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=3.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=4.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=5.0, duration=1.0)
create_chord(chord_track, ["G2", "B2", "D3"], start_time=6.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=7.0, duration=1.0)

seq.add_track(melody_track)
seq.add_track(chord_track)
seq.set_instrument("melody", BasicPiano())
seq.set_instrument("chords", BasicOrgan())

audio = seq.render()
display(play_sound(audio, "ステップ2：メロディ + コード伴奏"))

### ステップ3：ベースラインを追加する

In [ ]:
seq = Sequencer()

melody_track = Track(name="melody")
melody_notes = [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
]
create_simple_melody(melody_track, melody_notes, note_duration=0.5)

chord_track = Track(name="chords")
create_chord(chord_track, ["C3", "E3", "G3"], start_time=0.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=1.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=2.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=3.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=4.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=5.0, duration=1.0)
create_chord(chord_track, ["G2", "B2", "D3"], start_time=6.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=7.0, duration=1.0)

bass_track = Track(name="bass")
bass_notes = [
    ("C2", 100, 0.0, 1.0), ("C2", 100, 1.0, 1.0),
    ("F2", 100, 2.0, 1.0), ("C2", 100, 3.0, 1.0),
    ("F2", 100, 4.0, 1.0), ("C2", 100, 5.0, 1.0),
    ("G2", 100, 6.0, 1.0), ("C2", 100, 7.0, 1.0),
]
bass_track.add_notes(bass_notes)

seq.add_track(melody_track)
seq.add_track(chord_track)
seq.add_track(bass_track)
seq.set_instrument("melody", BasicPiano())
seq.set_instrument("chords", BasicOrgan())
seq.set_instrument("bass", BasicGuitar())

audio = seq.render()
display(play_sound(audio, "ステップ3：メロディ + コード + ベース"))

### ステップ4：ドラムパターンを追加する

BasicDrumのMIDIノート番号：36=キック、38=スネア、42=ハイハット

In [ ]:
seq = Sequencer()

melody_track = Track(name="melody")
create_simple_melody(melody_track, [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
], note_duration=0.5)

chord_track = Track(name="chords")
create_chord(chord_track, ["C3", "E3", "G3"], start_time=0.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=1.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=2.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=3.0, duration=1.0)
create_chord(chord_track, ["F3", "A3", "C4"], start_time=4.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=5.0, duration=1.0)
create_chord(chord_track, ["G2", "B2", "D3"], start_time=6.0, duration=1.0)
create_chord(chord_track, ["C3", "E3", "G3"], start_time=7.0, duration=1.0)

bass_track = Track(name="bass")
bass_track.add_notes([
    ("C2", 100, 0.0, 1.0), ("C2", 100, 1.0, 1.0),
    ("F2", 100, 2.0, 1.0), ("C2", 100, 3.0, 1.0),
    ("F2", 100, 4.0, 1.0), ("C2", 100, 5.0, 1.0),
    ("G2", 100, 6.0, 1.0), ("C2", 100, 7.0, 1.0),
])

drum_track = Track(name="drums")
drum_notes = []
for beat in range(16):
    t = beat * 0.5
    drum_notes.append((42, 80, t, 0.25))
    if beat % 2 == 0:
        drum_notes.append((36, 100, t, 0.25))
    if beat % 2 == 1:
        drum_notes.append((38, 90, t, 0.25))
drum_track.add_notes(drum_notes)

seq.add_track(melody_track)
seq.add_track(chord_track)
seq.add_track(bass_track)
seq.add_track(drum_track)
seq.set_instrument("melody", BasicPiano())
seq.set_instrument("chords", BasicOrgan())
seq.set_instrument("bass", BasicGuitar())
seq.set_instrument("drums", BasicDrum())

audio = seq.render()
display(play_sound(audio, "ステップ4：フルアレンジ"))

### ステップ5：エフェクトを適用して仕上げる

In [ ]:
# ステップ4と同じ構成 + リバーブ
seq = Sequencer()

melody_track = Track(name="melody")
create_simple_melody(melody_track, [
    "C4", "C4", "G4", "G4", "A4", "A4", "G4", None,
    "F4", "F4", "E4", "E4", "D4", "D4", "C4", None,
], note_duration=0.5)

chord_track = Track(name="chords")
for i, chord in enumerate([
    ["C3","E3","G3"], ["C3","E3","G3"], ["F3","A3","C4"], ["C3","E3","G3"],
    ["F3","A3","C4"], ["C3","E3","G3"], ["G2","B2","D3"], ["C3","E3","G3"],
]):
    create_chord(chord_track, chord, start_time=i*1.0, duration=1.0)

bass_track = Track(name="bass")
bass_track.add_notes([
    ("C2", 100, 0.0, 1.0), ("C2", 100, 1.0, 1.0),
    ("F2", 100, 2.0, 1.0), ("C2", 100, 3.0, 1.0),
    ("F2", 100, 4.0, 1.0), ("C2", 100, 5.0, 1.0),
    ("G2", 100, 6.0, 1.0), ("C2", 100, 7.0, 1.0),
])

drum_track = Track(name="drums")
drum_notes = []
for beat in range(16):
    t = beat * 0.5
    drum_notes.append((42, 80, t, 0.25))
    if beat % 2 == 0:
        drum_notes.append((36, 100, t, 0.25))
    if beat % 2 == 1:
        drum_notes.append((38, 90, t, 0.25))
drum_track.add_notes(drum_notes)

seq.add_track(melody_track)
seq.add_track(chord_track)
seq.add_track(bass_track)
seq.add_track(drum_track)
seq.set_instrument("melody", BasicPiano())
seq.set_instrument("chords", BasicOrgan())
seq.set_instrument("bass", BasicGuitar())
seq.set_instrument("drums", BasicDrum())

audio = seq.render()

reverb = Reverb(room_size=0.4, damping=0.5, wet_level=0.2)
audio_with_reverb = reverb.process(audio)
display(play_sound(audio_with_reverb, "ステップ5：完成版（リバーブ適用）"))

# 比較用に波形を表示
fig, axes = plt.subplots(2, 1, figsize=(12, 4))
t = np.arange(len(audio.data)) / sample_rate
axes[0].plot(t, audio.data, linewidth=0.5)
axes[0].set_title("エフェクトなし")
axes[0].set_ylabel("振幅")
t_rev = np.arange(len(audio_with_reverb.data)) / sample_rate
axes[1].plot(t_rev, audio_with_reverb.data, linewidth=0.5)
axes[1].set_title("リバーブ適用後")
axes[1].set_xlabel("時間 (秒)")
axes[1].set_ylabel("振幅")
plt.tight_layout()
plt.show()

## 🎯 例題2：「かえるの合唱」を輪唱（カノン）で実装

同じメロディを時間差で重ねる「輪唱（カノン）」は、シーケンサーの能力を試すよい題材です。

In [ ]:
frog_melody = [
    "C4", "D4", "E4", "F4", "E4", "D4", "C4", None,
    "E4", "F4", "G4", "A4", "G4", "F4", "E4", None,
]

note_dur = 0.4
round_delay = note_dur * 8  # 8拍遅れで次の声部が入る

seq = Sequencer()

# 声部1（ピアノ）
voice1 = Track(name="voice1")
create_simple_melody(voice1, frog_melody, note_duration=note_dur, start_time=0.0)

# 声部2（オルガン）— 8拍遅れ
voice2 = Track(name="voice2")
create_simple_melody(voice2, frog_melody, note_duration=note_dur, start_time=round_delay)

# 声部3（ギター）— 16拍遅れ
voice3 = Track(name="voice3")
create_simple_melody(voice3, frog_melody, note_duration=note_dur, start_time=round_delay * 2)

seq.add_track(voice1)
seq.add_track(voice2)
seq.add_track(voice3)
seq.set_instrument("voice1", BasicPiano())
seq.set_instrument("voice2", BasicOrgan())
seq.set_instrument("voice3", BasicGuitar())

total_dur = round_delay * 2 + note_dur * len(frog_melody) + 0.5
audio = seq.render(duration=total_dur)

reverb = Reverb(room_size=0.3, damping=0.5, wet_level=0.15)
audio = reverb.process(audio)

print(f"声部1（ピアノ）: 0.0秒から開始")
print(f"声部2（オルガン）: {round_delay:.1f}秒から開始")
print(f"声部3（ギター）: {round_delay * 2:.1f}秒から開始")
display(play_sound(audio, "かえるの合唱 — 3声の輪唱"))

## 🎵 楽譜からコードへの変換ガイド

### 音符の長さとプログラム上の秒数

テンポ120 BPM（1分間に120拍）の場合：

| 音符 | 名前 | 拍数 | 秒数 |
|------|------|------|------|
| 𝅝 | 全音符 | 4拍 | 2.0秒 |
| 𝅗𝅥 | 2分音符 | 2拍 | 1.0秒 |
| ♩ | 4分音符 | 1拍 | 0.5秒 |
| ♪ | 8分音符 | 0.5拍 | 0.25秒 |
| ♬ | 16分音符 | 0.25拍 | 0.125秒 |

**計算式**: `秒数 = (60 / BPM) * 拍数`

### 休符の扱い
`create_simple_melody` では `None` が休符。`add_notes` では音符を置かなければ自動的に無音。

### 音名とノート番号

| ド | レ | ミ | ファ | ソ | ラ | シ |
|----|----|----|------|----|----|-----|
| C4 | D4 | E4 | F4 | G4 | A4 | B4 |

- `#`（シャープ）: `"C#4"`, `"F#4"` など
- オクターブ: 数字が大きいほど高い音（C3 < C4 < C5）

In [ ]:
def beats_to_seconds(beats, bpm=120):
    """拍数を秒数に変換"""
    return beats * (60.0 / bpm)

bpm = 120
print(f"テンポ: {bpm} BPM")
print(f"  4分音符（1拍）: {beats_to_seconds(1, bpm):.3f}秒")
print(f"  8分音符（0.5拍）: {beats_to_seconds(0.5, bpm):.3f}秒")
print(f"  2分音符（2拍）: {beats_to_seconds(2, bpm):.3f}秒")
print(f"  付点4分音符（1.5拍）: {beats_to_seconds(1.5, bpm):.3f}秒")

bpm_slow = 100
print(f"\nテンポ: {bpm_slow} BPM")
print(f"  4分音符（1拍）: {beats_to_seconds(1, bpm_slow):.3f}秒")

## 🎵 音色設計リファレンス

### 楽器クラスを使う（簡単）

| 音色 | クラス | 特徴 |
|------|--------|------|
| ピアノ風 | `BasicPiano()` | 正弦波＋短めの減衰 |
| オルガン風 | `BasicOrgan()` | 矩形波＋持続的な音 |
| ギター風 | `BasicGuitar()` | のこぎり波＋プラック的な減衰 |
| ドラム風 | `BasicDrum()` | ノート番号で打楽器を指定（36=キック, 38=スネア, 42=ハイハット） |

### 波形＋エンベロープで自作する（応用）

In [ ]:
freq = note_to_frequency(note_name_to_number("C4"))
dur = 1.5

# 1. ピアノ風: 正弦波 + 短い減衰
piano_sound = sine_wave(freq, dur) * adsr(dur, attack=0.01, decay=0.3, sustain=0.3, release=0.2)
display(play_sound(piano_sound, "ピアノ風: sine + 短い減衰"))

# 2. オルガン風: 矩形波 + 持続的エンベロープ
organ_sound = square_wave(freq, dur) * 0.3 * adsr(dur, attack=0.05, decay=0.1, sustain=0.8, release=0.2)
display(play_sound(organ_sound, "オルガン風: square + 持続的エンベロープ"))

# 3. ギター風: のこぎり波 + プラック減衰
guitar_sound = sawtooth_wave(freq, dur) * 0.3 * adsr(dur, attack=0.005, decay=0.4, sustain=0.1, release=0.3)
display(play_sound(guitar_sound, "ギター風: sawtooth + プラック減衰"))

# 4. 8bit風: 矩形波 + シンプルなエンベロープ
bit8_sound = square_wave(freq, dur) * 0.3 * adsr(dur, attack=0.001, decay=0.05, sustain=0.6, release=0.05)
display(play_sound(bit8_sound, "8bit風: square + シンプルエンベロープ"))

# 5. パッド風: 正弦波 + 長いアタック/リリース + リバーブ
pad_sound = sine_wave(freq, dur) * adsr(dur, attack=0.5, decay=0.2, sustain=0.7, release=0.5)
pad_sound = Reverb(room_size=0.7, wet_level=0.4).process(pad_sound)
display(play_sound(pad_sound, "パッド風: sine + 長いアタック + リバーブ"))

## 🎯 自分の曲を選んで実装しよう

ここからが本番です。好きな曲を1曲選び、上の例題と同じ手順で実装してください。

### 曲選びのヒント
- 童謡や唱歌はメロディがシンプルで取り組みやすい
- ゲーム音楽の8bit曲は矩形波と相性がよい
- テンポが遅い曲の方が打ち込みやすい
- まずは8〜16小節程度の短い部分から始める

### ステップ1：曲の情報を記入する

In [ ]:
# TODO: 自分の曲の情報を記入
song_title = "曲名をここに"  # 例: "チューリップ"
song_bpm = 120
beat_duration = 60.0 / song_bpm

print(f"曲: {song_title}")
print(f"テンポ: {song_bpm} BPM")
print(f"1拍 = {beat_duration:.3f}秒")
print(f"1小節（4拍） = {beat_duration * 4:.3f}秒")

### ステップ2：メロディを打ち込む

In [ ]:
seq = Sequencer()

# TODO: メロディを入力
melody_track = Track(name="melody")
my_melody = [
    # TODO: 音名を入力（例: "C4", "D4", "E4", ...）
    # 休符は None
]
create_simple_melody(melody_track, my_melody, note_duration=beat_duration)

seq.add_track(melody_track)
seq.set_instrument("melody", BasicPiano())  # TODO: 楽器を選ぶ

# audio = seq.render()
# display(play_sound(audio, "メロディの確認"))

### ステップ3：コード伴奏を追加する

In [ ]:
# TODO: コード伴奏を追加
chord_track = Track(name="chords")

# create_chord(chord_track, ["C3", "E3", "G3"], start_time=0.0, duration=beat_duration * 4)
# create_chord(chord_track, ["F3", "A3", "C4"], start_time=beat_duration * 4, duration=beat_duration * 4)

seq.add_track(chord_track)
seq.set_instrument("chords", BasicOrgan())

# audio = seq.render()
# display(play_sound(audio, "メロディ + コードの確認"))

### ステップ4：ベースラインを追加する

In [ ]:
# TODO: ベースラインを追加（コードのルート音を使うのが基本）
bass_track = Track(name="bass")

# bass_track.add_notes([
#     ("C2", 100, 0.0, beat_duration * 4),
#     ("F2", 100, beat_duration * 4, beat_duration * 4),
# ])

seq.add_track(bass_track)
seq.set_instrument("bass", BasicGuitar())

# audio = seq.render()
# display(play_sound(audio, "メロディ + コード + ベースの確認"))

### ステップ5：ドラムパターンを追加する

In [ ]:
# TODO: ドラムパターンを追加
# ドラム: 36=キック, 38=スネア, 42=ハイハット
drum_track = Track(name="drums")

# total_beats = ...
# drum_notes = []
# for beat in range(total_beats):
#     t = beat * beat_duration
#     drum_notes.append((42, 80, t, 0.1))
#     if beat % 4 == 0:
#         drum_notes.append((36, 100, t, 0.2))
#     if beat % 4 == 2:
#         drum_notes.append((38, 90, t, 0.2))
# drum_track.add_notes(drum_notes)

seq.add_track(drum_track)
seq.set_instrument("drums", BasicDrum())

# audio = seq.render()
# display(play_sound(audio, "フルアレンジの確認"))

### ステップ6：エフェクトを適用する

In [ ]:
# TODO: エフェクトを適用して仕上げる

# reverb = Reverb(room_size=0.4, damping=0.5, wet_level=0.2)
# audio = reverb.process(audio)

# delay = Delay(delay_time=0.3, feedback=0.2, wet_level=0.15)
# audio = delay.process(audio)

# lpf = LowPassFilter(cutoff_freq=3000)
# audio = lpf.process(audio)

# display(play_sound(audio, "最終版"))

### ステップ7：最終ミックス（まとめて実行）

上のステップで確認しながら作った内容を、1つのセルにまとめて最終版を作りましょう。

In [ ]:
# ===== 最終ミックス =====
# TODO: 上のステップで作った全パートをまとめる

seq = Sequencer()

# --- メロディ ---
melody_track = Track(name="melody")
# TODO: メロディの音符をここに

# --- コード ---
chord_track = Track(name="chords")
# TODO: コード進行をここに

# --- ベース ---
bass_track = Track(name="bass")
# TODO: ベースラインをここに

# --- ドラム ---
drum_track = Track(name="drums")
# TODO: ドラムパターンをここに

# --- トラック登録と楽器設定 ---
seq.add_track(melody_track)
seq.add_track(chord_track)
seq.add_track(bass_track)
seq.add_track(drum_track)
seq.set_instrument("melody", BasicPiano())
seq.set_instrument("chords", BasicOrgan())
seq.set_instrument("bass", BasicGuitar())
seq.set_instrument("drums", BasicDrum())

# --- レンダリング ---
# audio = seq.render()

# --- エフェクト ---
# TODO: 必要なエフェクトを適用

# --- 再生 ---
# display(play_sound(audio, f"完成: {song_title}"))

## 🎤 発表シート

### 作品情報

- **選んだ曲**: （曲名を記入）
- **選んだ理由**: （なぜこの曲を選んだか）

### 使用した技術要素

- [ ] 波形生成（sine, square, sawtooth, triangle）
- [ ] ADSRエンベロープ
- [ ] シーケンサーによるマルチトラック構成
- [ ] 楽器クラス（BasicPiano, BasicOrgan, BasicGuitar, BasicDrum）
- [ ] コード（和音）
- [ ] エフェクト（リバーブ、ディレイ、フィルターなど）

### 音色設計で工夫した点

（どの楽器にどの音色を使い、なぜそれを選んだか）

### 原曲との違いと、その理由

（技術的制約による違い、意図的に変更した部分など）

### 実装で苦労した点と解決方法

（つまずいたポイントと、どう解決したか）

## 🏆 ボーナスチャレンジ

### チャレンジ1：同じ曲を違うジャンルにアレンジ

- **8bit風**: 全パートを `square_wave` で、エフェクトなし
- **アンビエント風**: `sine_wave` + 長いアタック/リリース + 深いリバーブ
- **ロック風**: `sawtooth_wave` + ドラムを強調

In [ ]:
# チャレンジ1：アレンジ版
# TODO: 同じメロディを違う音色・エフェクトで再実装

### チャレンジ2：パラメータ化して演奏条件を変える

In [ ]:
def render_song(bpm=120, transpose=0):
    """曲をレンダリングする関数

    Args:
        bpm: テンポ（BPM）
        transpose: 移調量（半音単位、例: +2 で全音上げ）
    """
    bt = 60.0 / bpm  # noqa: F841

    seq = Sequencer()
    melody_track = Track(name="melody")

    # TODO: 自分の曲のメロディを入れる
    # transposeを適用する例:
    # original_notes = ["C4", "D4", "E4", "F4"]
    # for i, note_name in enumerate(original_notes):
    #     midi_num = note_name_to_number(note_name) + transpose
    #     melody_track.add_note(midi_num, 100, i * bt, bt)

    seq.add_track(melody_track)
    seq.set_instrument("melody", BasicPiano())
    return seq.render()

# 通常版
# display(play_sound(render_song(bpm=120, transpose=0), "通常 (120 BPM)"))
# display(play_sound(render_song(bpm=80, transpose=0), "スロー (80 BPM)"))
# display(play_sound(render_song(bpm=120, transpose=4), "キー +4"))

## 📚 コース振り返り

### 学んだ技術の積み上げ

| レッスン | テーマ | 学んだこと |
|----------|--------|------------|
| 01 | 音の基礎と正弦波 | 周波数・振幅・位相、正弦波の生成 |
| 02 | エンベロープとADSR | 音の時間変化、ADSRパラメータ |
| 03 | 周波数分析 | FFT、倍音構造、スペクトログラム |
| 04 | フィルターと音色設計 | ローパスフィルター、楽器音の作成 |
| 05 | エフェクトとダイナミクス | リバーブ、ディレイ、コーラス |
| 06 | MIDIとシーケンサー | ノート、トラック、マルチパート |
| 07 | 最終プロジェクト | 全技術を統合して1曲を再現 |

### このコースで身についたこと

- **音の物理**: 音が波であること、周波数が音の高さを決めること
- **信号処理の基礎**: デジタル音声がどのように生成・加工されるか
- **プログラミングによる実装力**: 理論を実際のコードに落とし込む力

### 次のステップ

- **信号処理をさらに学ぶ**: FIR/IIRフィルターの設計、窓関数、周波数領域での処理
- **シンセサイザーの仕組み**: FM合成、加算合成、減算合成、ウェーブテーブル合成
- **音声プログラミングツール**: SuperCollider, Pure Data, Max/MSP
- **Python音声ライブラリ**: librosa（音声分析）, pydub（音声編集）, pedalboard（エフェクト）

---

お疲れさまでした！全レッスン完了です。

このコースでは「音とは何か」という物理の話から始まり、信号処理の基礎を学び、最終的にプログラムで1曲を再現するところまで到達しました。ここで学んだ知識は、音楽制作ソフトの仕組みを理解したり、音声処理のアプリケーションを開発したりする際の土台になります。